# 문서 벡터를 저장하고 다시 열기

문서와 벡터를 Chroma에 저장하고, 같은 저장소를 다시 열어 재사용합니다. 전체 검색과 정책 자료만 고르는 검색을 비교합니다.


### 준비. 환경과 저장 위치

- **벡터 저장소**: 문서 벡터와 원문·부가 정보를 보관하고 유사한 문서를 찾는 저장소.
- **저장 경로**: `DATABASE_DIR`는 디스크 자료의 위치. 같은 자료를 다시 열 때 같은 경로 사용.


In [4]:
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()

LESSON_DIR = Path.cwd()
DATABASE_DIR = LESSON_DIR / ".chroma-lab" / "center-policy-v1"

QUESTION = "강의를 취소하면 결제한 돈을 돌려받을 수 있나요?"
EMBEDDING_MODEL = "text-embedding-3-small"


## 1. 문서와 벡터 저장·재사용

### 1-1. 짧은 문서와 고정 ID

- **`Document`**: 본문 `page_content`, 부가 정보 `metadata`, 식별자 `id`를 담은 문서 객체.
- **고정 ID**: 저장할 문서 항목을 구분하는 값. 같은 파일명의 여러 본문도 별도 ID로 식별.
- **metadata**: 출처와 주제 같은 부가 정보. 뒤의 필터가 이 값을 사용.


In [5]:
DOCUMENTS = [
    Document(id="refund", page_content="수업 시작 24시간 전까지 취소하면 전액 환불됩니다.", metadata={"source": "center_policy.txt", "topic": "policy"}),
    Document(id="hours", page_content="토요일 실습실은 오전 10시부터 오후 5시까지 운영합니다.", metadata={"source": "center_policy.txt", "topic": "policy"}),
    Document(id="rag", page_content="RAG는 외부 문서를 검색해 모델 요청에 넣는 방식입니다.", metadata={"source": "course_guide.txt", "topic": "lesson"}),
]
print(type(DOCUMENTS).__name__, len(DOCUMENTS))
for document in DOCUMENTS:
    print(document.id, document.metadata, document.page_content)

list 3
refund {'source': 'center_policy.txt', 'topic': 'policy'} 수업 시작 24시간 전까지 취소하면 전액 환불됩니다.
hours {'source': 'center_policy.txt', 'topic': 'policy'} 토요일 실습실은 오전 10시부터 오후 5시까지 운영합니다.
rag {'source': 'course_guide.txt', 'topic': 'lesson'} RAG는 외부 문서를 검색해 모델 요청에 넣는 방식입니다.


### 1-2. 같은 컬렉션 열기

- **Chroma**: 디스크에 문서·벡터를 보관할 수 있는 벡터 데이터베이스.
- **컬렉션(collection)**: 저장소 안의 문서 묶음. 폴더 경로와 컬렉션 이름을 함께 구분.
- **`embedding_function`**: 문서·질문을 벡터로 바꿀 임베딩 객체 연결.
- **`get()` 반환값**: 항목들을 담은 딕셔너리. `ids`는 ID 문자열 리스트이며 유사도 검색과 구분.


In [6]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, request_timeout=20, max_retries=0)
store = Chroma(collection_name="center-policy-v1", embedding_function=embeddings, persist_directory=str(DATABASE_DIR))
stored = store.get(limit=1)
print(type(stored).__name__, stored["ids"])

dict []


### 1-3. 최초 저장 또는 재사용

- **`add_documents()`**: 문서를 임베딩하고 ID·본문·metadata와 함께 저장.
- **영속 저장**: `persist_directory`를 지정한 저장 폴더의 자료는 커널 종료 후에도 남음.
- **재사용 조건**: ID가 하나 이상이면 추가 생략. 모든 문서의 최신성·완전성을 검사하는 조건은 아님.


In [ ]:
# 고정된 실습 자료 존재 확인
if stored["ids"]:
    print("상태: 저장한 문서와 벡터 재사용")
else:
    ids = [str(document.id) for document in DOCUMENTS]

    # 문서 목록과 식별자를 저장소에 전달
    # 연결한 임베딩 도구가 세 본문을 벡터로 바꾸고, 본문/metadata/벡터가 각각의 ID와 함께 보관된다
    store.add_documents(DOCUMENTS, ids=ids)
    print("상태: 문서와 벡터를 처음 저장함")

### 1-4. 실제로 저장한 내용 조회

- **저장 내용**: 반환 딕셔너리의 `ids`, `documents`, `metadatas`는 서로 대응하는 리스트.
- **`zip()`**: 같은 위치의 ID·본문·metadata를 튜플로 묶어 순회.
- **조회와 검색**: 저장된 항목 조회는 질문 임베딩을 요청하지 않음.


In [11]:
saved = store.get(include=["documents", "metadatas"])
print("문서 수:", len(saved["ids"]))
for item in zip(saved["ids"], saved["documents"], saved["metadatas"]):
    print(item)

문서 수: 3
('refund', '수업 시작 24시간 전까지 취소하면 전액 환불됩니다.', {'topic': 'policy', 'source': 'center_policy.txt'})
('hours', '토요일 실습실은 오전 10시부터 오후 5시까지 운영합니다.', {'source': 'center_policy.txt', 'topic': 'policy'})
('rag', 'RAG는 외부 문서를 검색해 모델 요청에 넣는 방식입니다.', {'source': 'course_guide.txt', 'topic': 'lesson'})


## 2. 검색 결과와 metadata 필터

### 2-1. 전체 문서에서 검색

- **Retriever**: 저장소를 검색하는 공통 실행 인터페이스. `as_retriever()`로 구성.
- **`invoke(QUESTION)`**: 현재 질문을 임베딩해 저장 벡터와 비교한 뒤 문서 후보 반환.
- **`k=2`**: 가져올 최대 문서 수.


In [13]:
all_retriever = store.as_retriever(search_kwargs={"k": 2})
all_result = all_retriever.invoke(QUESTION)

### 2-2. 검색 반환값 읽기

- **반환 타입**: `list[Document]` — `get()`이 반환한 딕셔너리와 구분.
- **원문 검색 결과**: 저장된 본문·metadata를 읽는 단계. 모델이 생성한 답변은 아님.
- **근거 확인**: 출처 이름과 질문에 필요한 본문 내용은 별도로 확인.


In [15]:
print(type(all_result).__name__, len(all_result))
for document in all_result:
    print(document.id, document.metadata, document.page_content)

list 2
refund {'topic': 'policy', 'source': 'center_policy.txt'} 수업 시작 24시간 전까지 취소하면 전액 환불됩니다.
rag {'topic': 'lesson', 'source': 'course_guide.txt'} RAG는 외부 문서를 검색해 모델 요청에 넣는 방식입니다.


### 2-3. 정책 필터로 검색하고 전체 결과와 비교

- **metadata 필터**: `topic="policy"`인 문서만 검색 대상으로 선택.
- **필터와 개수**: filter는 대상 조건, k는 그 안에서 반환할 최대 문서 수.
- **비교 범위**: 정책 종류에 속한다는 사실과 환불 질문에 답할 근거라는 사실은 구분.


In [16]:
policy_retriever = store.as_retriever(search_kwargs={"k": 2, "filter": {"topic": "policy"}})

policy_results = policy_retriever.invoke(QUESTION)
for document in policy_results:
    print(document.id, document.metadata, document.page_content)

refund {'source': 'center_policy.txt', 'topic': 'policy'} 수업 시작 24시간 전까지 취소하면 전액 환불됩니다.
hours {'source': 'center_policy.txt', 'topic': 'policy'} 토요일 실습실은 오전 10시부터 오후 5시까지 운영합니다.
